# 00 — First principles of customer support automation

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/01_customer_support_agent/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and NLP familiarity
- Understanding of embeddings and cosine similarity

**What you will learn**:
- Why customer support decomposes into four sub-problems: intent classification, knowledge retrieval, response generation, and escalation judgment
- Why keyword/regex chatbots fail on real user language and how semantic similarity solves this
- How confidence calibration and cost-aware escalation prevent catastrophic wrong answers
- Why multi-turn conversation state is essential for real support interactions
- The market economics that make AI support a $12 B+ opportunity

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "sentence-transformers>=2.7.0" "numpy>=1.24.0" "scikit-learn>=1.3.0"

import re
import numpy as np
from typing import Optional
from sklearn.metrics.pairwise import cosine_similarity

## Section 1 — What is customer support?

Customer support is not a single task — it is **four sub-problems** running in sequence on every user message:

| # | Sub-problem | Question it answers | Example |
|---|-------------|---------------------|--------|
| 1 | **Intent classification** | *What does the user want?* | "Where's my order?" → `order_status` |
| 2 | **Knowledge retrieval** | *What information do we need?* | Fetch shipping policy + order details |
| 3 | **Response generation** | *How do we phrase the answer?* | Polite, concise, on-brand reply |
| 4 | **Escalation judgment** | *Can we handle this, or should a human?* | Low confidence → route to agent |

Every support system — from a phone tree to GPT-4 — implements these four steps. The only question is *how well*.

Let's model this with a simple data structure:

In [ ]:
# Model a support interaction as four explicit sub-problems

from dataclasses import dataclass


@dataclass
class SupportQuery:
    """A single customer support interaction broken into sub-problems."""
    user_message: str
    intent: Optional[str] = None
    retrieved_docs: Optional[list[str]] = None
    response: Optional[str] = None
    confidence: Optional[float] = None
    escalate: bool = False


# Concrete examples showing the four components
examples = [
    SupportQuery(
        user_message="Where is my order #12345?",
        intent="order_status",
        retrieved_docs=["Orders ship within 2-3 business days."],
        response="Let me look up order #12345 for you.",
        confidence=0.92,
        escalate=False,
    ),
    SupportQuery(
        user_message="Your product broke after one day and I want a refund NOW",
        intent="complaint",
        retrieved_docs=["30-day return policy for defective items."],
        response="I'm sorry about that. You're eligible for a full refund.",
        confidence=0.75,
        escalate=False,
    ),
    SupportQuery(
        user_message="I was charged twice and my bank is involved",
        intent="billing",
        retrieved_docs=["Duplicate charge resolution process."],
        response=None,  # too risky to auto-respond
        confidence=0.35,
        escalate=True,  # route to human
    ),
]

for ex in examples:
    action = "ESCALATE" if ex.escalate else f"RESPOND (conf={ex.confidence:.2f})"
    print(f"Message:  {ex.user_message}")
    print(f"Intent:   {ex.intent}")
    print(f"Action:   {action}")
    print("-" * 60)

## Section 2 — Why rule-based chatbots fail

Before LLMs, chatbots matched user messages with **keywords and regex patterns**. This works for the exact phrasing the developer anticipated — and breaks on everything else.

Let's build a keyword chatbot and watch it fail:

In [ ]:
# A simple keyword/regex chatbot

RULES: list[tuple[str, str]] = [
    (r"order.*(status|track|where)", "Your order is being processed. Check your email for tracking."),
    (r"(return|refund)", "Our return policy allows returns within 30 days."),
    (r"(price|cost|how much)", "Please visit our pricing page for current prices."),
    (r"(hello|hi|hey)", "Hello! How can I help you today?"),
]


def keyword_chatbot(message: str) -> str:
    """Match the first rule whose pattern appears in the message."""
    message_lower = message.lower()
    for pattern, response in RULES:
        if re.search(pattern, message_lower):
            return response
    return "I'm sorry, I don't understand. Please contact support@example.com."


# Test on exact phrasing — works fine
test_exact = [
    "Where is my order?",
    "I want a refund",
    "Hi there!",
]
print("=== Exact phrasing (works) ===")
for msg in test_exact:
    print(f"User: {msg}")
    print(f"Bot:  {keyword_chatbot(msg)}\n")

In [ ]:
# Now test on paraphrases — the chatbot fails

test_paraphrases = [
    "My order hasn't arrived",
    "Where's my package?",
    "Still waiting on my delivery",
    "It's been 5 days and nothing showed up",
    "Can I send this back?",
    "I'd like my money back please",
]

print("=== Paraphrases (fails) ===")
for msg in test_paraphrases:
    resp = keyword_chatbot(msg)
    hit = "MISS" if "don't understand" in resp else "HIT"
    print(f"[{hit}] User: {msg}")
    print(f"       Bot:  {resp}\n")

### Semantic similarity to the rescue

All of these paraphrases **mean the same thing**. Keyword matching cannot see this, but embedding-based cosine similarity can. Let's prove it:

In [ ]:
# Demonstrate semantic similarity with sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Three paraphrases of the same intent
queries = [
    "My order hasn't arrived",
    "Where's my package?",
    "Still waiting on my delivery",
]
embeddings = model.encode(queries)

print("Cosine similarity between paraphrases:")
sim_matrix = cosine_similarity(embeddings)
for i in range(len(queries)):
    for j in range(i + 1, len(queries)):
        print(f"  '{queries[i]}' <-> '{queries[j]}': {sim_matrix[i][j]:.3f}")

# Now compare to an unrelated query
unrelated = model.encode(["What's the weather like?"])
print("\nSimilarity to unrelated query:")
for i, q in enumerate(queries):
    sim = cosine_similarity([embeddings[i]], unrelated)[0][0]
    print(f"  '{q}' <-> 'What's the weather like?': {sim:.3f}")

## Section 3 — The confidence calibration problem

In customer support, **answering wrong is far worse than saying "I don't know."**

- A wrong billing answer can cost the company money and trust
- An unnecessary escalation just adds a small delay

This asymmetry means we need a **cost matrix** to guide the agent's decisions:

| Scenario | Cost |
|----------|------|
| Correct self-service answer | **+5** (saved a human agent's time) |
| Escalation to human | **+1** (small cost, safe fallback) |
| Wrong answer delivered | **−10** (customer frustration, possible churn) |

The implication: **it's better to escalate 5 uncertain queries than to give 1 wrong answer.**

In [ ]:
# Cost-aware confidence scoring

COST_CORRECT = 5
COST_ESCALATE = 1
COST_WRONG = -10


def expected_value(confidence: float) -> dict[str, float]:
    """Calculate the expected value of responding vs escalating."""
    ev_respond = confidence * COST_CORRECT + (1 - confidence) * COST_WRONG
    ev_escalate = COST_ESCALATE  # always the same regardless of confidence
    return {"ev_respond": ev_respond, "ev_escalate": ev_escalate}


# Find the break-even confidence threshold
print("Confidence | EV(Respond) | EV(Escalate) | Decision")
print("-" * 58)
for conf in [0.3, 0.5, 0.6, 0.7, 0.73, 0.74, 0.8, 0.9, 0.95]:
    ev = expected_value(conf)
    decision = "RESPOND" if ev["ev_respond"] > ev["ev_escalate"] else "ESCALATE"
    print(f"    {conf:.2f}   |    {ev['ev_respond']:+.2f}    |     {ev['ev_escalate']:+.2f}     | {decision}")

# Exact threshold: solve conf * 5 + (1-conf) * (-10) = 1 → 15*conf = 11 → conf = 11/15
threshold = (COST_ESCALATE - COST_WRONG) / (COST_CORRECT - COST_WRONG)
print(f"\nBreak-even threshold: {threshold:.4f}")
print(f"Rule: only auto-respond when confidence > {threshold:.2f}")

In [ ]:
# Simulate 100 queries with varying confidence and show total value

np.random.seed(42)
n_queries = 100
confidences = np.random.beta(5, 2, size=n_queries)  # skewed toward higher confidence
is_correct = np.random.random(n_queries) < confidences  # actual correctness

# Strategy 1: always respond (no escalation)
total_always_respond = sum(
    COST_CORRECT if correct else COST_WRONG
    for correct in is_correct
)

# Strategy 2: escalate when confidence < threshold
total_smart_escalate = 0
escalated = 0
for conf, correct in zip(confidences, is_correct):
    if conf >= threshold:
        total_smart_escalate += COST_CORRECT if correct else COST_WRONG
    else:
        total_smart_escalate += COST_ESCALATE
        escalated += 1

print(f"Strategy: Always respond     → Total value: {total_always_respond:+d}")
print(f"Strategy: Smart escalation   → Total value: {total_smart_escalate:+.0f}")
print(f"  (escalated {escalated}/{n_queries} queries)")
print(f"  Improvement: {total_smart_escalate - total_always_respond:+.0f} points")

## Section 4 — Conversation state and multi-turn context

Real support conversations span multiple turns. Users provide information **incrementally**:

```
Turn 1: "I have a problem with my order"
Turn 2: "It was the blue wireless headphones"
Turn 3: "I ordered them last Tuesday"
Turn 4: "The left earpiece doesn't work"
```

By turn 4, the agent needs to remember *all* prior context. Without state management, each turn is processed in isolation — and "the left earpiece" has no referent.

In [ ]:
# Simple context accumulator for multi-turn conversations

from dataclasses import dataclass, field


@dataclass
class ConversationState:
    """Tracks entities and context across conversation turns."""
    history: list[dict[str, str]] = field(default_factory=list)
    entities: dict[str, str] = field(default_factory=dict)

    def add_turn(self, role: str, message: str) -> None:
        self.history.append({"role": role, "message": message})

    def extract_entities(self, message: str) -> dict[str, str]:
        """Extract key entities from a message using simple patterns."""
        found: dict[str, str] = {}
        # Order number
        order_match = re.search(r"#?(\d{4,})", message)
        if order_match:
            found["order_id"] = order_match.group(1)
        # Product mentions
        products = ["headphones", "speaker", "charger", "cable", "keyboard"]
        for p in products:
            if p in message.lower():
                found["product"] = p
        # Color
        colors = ["blue", "red", "black", "white", "silver"]
        for c in colors:
            if c in message.lower():
                found["color"] = c
        return found

    def process_message(self, message: str) -> str:
        """Process a user message and return accumulated context summary."""
        self.add_turn("user", message)
        new_entities = self.extract_entities(message)
        self.entities.update(new_entities)
        return f"Entities so far: {self.entities}"


# Simulate a multi-turn conversation
state = ConversationState()
turns = [
    "I have a problem with my order",
    "It was the blue wireless headphones",
    "Order number is #78432",
    "The left earpiece doesn't work",
]

print("=== Multi-turn context accumulation ===")
for i, msg in enumerate(turns, 1):
    summary = state.process_message(msg)
    print(f'Turn {i}: "{msg}"')
    print(f"  → {summary}")
    print()

print("Final context available to agent:")
print(f"  Entities: {state.entities}")
print(f"  History:  {len(state.history)} turns")

## Section 5 — Market analysis: why this is a $12 B+ opportunity

Customer support automation is one of the largest LLM application markets. Let's look at the economics:

| Metric | Human agent | AI agent |
|--------|-------------|----------|
| Cost per ticket | $6 – $12 | $0.50 – $2.00 |
| Availability | Business hours | 24/7 |
| Ramp-up time | 2–4 weeks training | Instant deployment |
| Consistency | Varies by agent | Deterministic (given same context) |
| Scalability | Linear (hire more) | Sub-linear (compute) |

**Case study — Klarna**: In 2024, Klarna reported their AI assistant handled 2.3 million conversations in its first month, doing the work of 700 full-time agents. They estimated **$40 M/year** in savings.

**The math**: If a company handles 100,000 tickets/month at $8/ticket = $800,000/month. Automating 60% at $1/ticket saves **$420,000/month**.

In [ ]:
# Unit economics calculator


def support_economics(
    monthly_tickets: int,
    human_cost_per_ticket: float,
    ai_cost_per_ticket: float,
    automation_rate: float,
) -> dict[str, float]:
    """Calculate monthly savings from AI support automation."""
    human_total = monthly_tickets * human_cost_per_ticket
    ai_tickets = int(monthly_tickets * automation_rate)
    human_tickets = monthly_tickets - ai_tickets
    blended_total = (ai_tickets * ai_cost_per_ticket) + (human_tickets * human_cost_per_ticket)
    savings = human_total - blended_total
    return {
        "human_only_cost": human_total,
        "blended_cost": blended_total,
        "monthly_savings": savings,
        "annual_savings": savings * 12,
        "cost_reduction_pct": (savings / human_total) * 100,
    }


# Scenario: mid-size e-commerce company
result = support_economics(
    monthly_tickets=100_000,
    human_cost_per_ticket=8.0,
    ai_cost_per_ticket=1.0,
    automation_rate=0.60,
)

print("=== Support automation economics ===")
print(f"Human-only cost:    ${result['human_only_cost']:>12,.0f} /month")
print(f"Blended cost:       ${result['blended_cost']:>12,.0f} /month")
print(f"Monthly savings:    ${result['monthly_savings']:>12,.0f}")
print(f"Annual savings:     ${result['annual_savings']:>12,.0f}")
print(f"Cost reduction:     {result['cost_reduction_pct']:.1f}%")

## Exercises

1. **Extend the keyword chatbot** — Add 5 more rules to `keyword_chatbot` covering shipping, warranty, and account issues. Then write 3 paraphrases for each that the chatbot *still* cannot handle. What does this tell you about the scalability of rule-based approaches?

2. **Tune the escalation threshold** — Change the cost matrix to `COST_CORRECT=3, COST_WRONG=-20, COST_ESCALATE=2` (a high-stakes domain like healthcare). Recalculate the break-even threshold. How does the threshold shift, and what does this mean for agent behaviour?

3. **Build a better entity extractor** — Extend `ConversationState.extract_entities` to handle dates ("last Tuesday", "on March 5th"), email addresses, and phone numbers. Test on a 6-turn conversation that includes all these entity types.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Customer support = intent classification + knowledge retrieval + response generation + escalation judgment |
| 2 | Keyword chatbots fail because users express the same intent in dozens of different phrasings |
| 3 | Semantic similarity (embeddings + cosine) captures meaning regardless of surface form |
| 4 | Confidence calibration with a cost matrix prevents the worst failure mode: confident wrong answers |
| 5 | Multi-turn state management is essential — users reveal information incrementally across turns |
| 6 | The economics are compelling: AI support costs 5–10× less per ticket than human agents |

## What's Next

- **Next**: [01_architecture.ipynb](./01_architecture.ipynb) — Design the RAG pipeline, intent router, and confidence gate
- **Related**: [02_build.ipynb](./02_build.ipynb) — Build the full support agent end-to-end

## References & Further Reading

1. [Klarna, "Klarna AI assistant handles two-thirds of customer service chats in its first month," 2024](https://www.klarna.com/international/press/klarna-ai-assistant-handles-two-thirds-of-customer-service-chats-in-its-first-month/) — Case study on AI support at scale
2. [Reimers & Gurevych, "Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks," 2019](https://arxiv.org/abs/1908.10084) — Foundation for semantic similarity in support
3. [Gartner, "Market Guide for Virtual Customer Assistants," 2023](https://www.gartner.com/en/documents/4023015) — Market sizing and vendor landscape for AI support